# Quantization and Normalization

In this notebook I will explore the mechanics behind quantization

In [129]:
import os
import sys
import pandas as pd
import json
from IPython.display import display, JSON
import numpy as np

sys.path.append("..")
sys.path.append("../code")

import torch

from dataset import PointCloudEmbeddingSequenceDataset

#### Adding Attributes (obj.name = "Said")
- In Python, instances have an internal dictionary (```__dict__```) that stores attributes.
- When you do ```obj.name = "Said"```, Python internally adds an entry to ```obj.__dict__``` like this:
```obj.__dict__["name"] = "Said"```
- When you access ```obj.name```, Python checks obj.__dict__ and returns "Said".

In [9]:
class Saver():
    pass

def construct_json_path(cad_seq_path):
    json_path = cad_seq_path.replace("cad_vec", "cad_json").replace("h5", "json")
    return json_path

def show_json():
    json_path = save.json_path
    with open(json_path, "r") as file:
        data = json.load(file)
    display(JSON(data))
    return data

def show_sample(idx, dataset):
    pc, lat_rep, cad_seq = dataset[idx]
    pc_path = dataset.get_cad_seq_path(idx)
    
    seq_length = list(cad_seq[:, 0]).index(3) + 3
    commands = cad_seq[:seq_length, 0]
    arguments = cad_seq[:seq_length,1:]

    command_list = torch.repeat_interleave(commands, 16).tolist()
    arg_list = []
    for command_args in arguments:
        arg_list += command_args.tolist()

    filtered_lists = [[x for x, trgt_arg in zip(lst, arg_list) if trgt_arg != -1] for lst in [command_list, arg_list]]
    command_list, arg_list = filtered_lists

    print("CAD-sequence path: ", pc_path)
    json_path = construct_json_path(pc_path)
    save.json_path = json_path
    print("Json path: ", json_path)
    df = pd.DataFrame(list(zip(command_list, arg_list)), columns=['command', 'argument'])
    return df

In [3]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")
save = Saver()

Dataset contains 8038 samples.


In [173]:
idx = 3

Below is a target vectorized CAD-sequence. In the following cells I will scrutinize the transformation from json data to this format.

In [174]:
show_sample(idx, dataset)

CAD-sequence path:  ../data/cad_vec/0023/00239323.h5
Json path:  ../data/cad_json/0023/00239323.json


,command,argument
0,1,176
1,1,128
2,1,128
3,1,1
4,0,176
5,0,199
6,1,128
7,1,199
8,1,128
9,1,1


In [175]:
json_data = show_json()

<IPython.core.display.JSON object>

The json file contains three dicts:
- **entities**: Sketches and Extrusion
    - Sketch:
        - Transform object with origin and x-, y- and z-axis vectors.
        - Profile object with all curves it is made out of. Each curve can be Arc, Circle or Line and the respective args are listed.
    - Extrusion:
         - References the Sketch under profiles and the extrusion arguments extent one, extent two, operation and extent type
- **properties**: Bounding box
    - Contains maximum and minimum point
- **sequence**: Describes the sequence of sketch and extrude operations by referencing the entities

The pipeline from json to vector representation comprises the normalization and quantization. Lets look at this now.

1. ```json2vec.py```: Loads the json files and creates a ```CADSequence``` object per json file
    - ```CADSequence``` has a staticmethod to create the class from a dict which is the json file. All ```from_dict()``` methods will be implemented as ```get_()``` methods here, but the class definition will stay the same.
    - ```CADSequence``` searches for the all the Extrude operations in the sequence, creates an ```Extrude``` object for each, and concatenates them in a list.
2. 

In [176]:
from models.DeepCAD.cadlib.extrude import CADSequence, Extrude
from models.DeepCAD.cadlib.sketch import Profile, Loop
from models.DeepCAD.cadlib.curves import Line, Arc, Circle

In [177]:
def get_CADSequence(all_stat):
    seq = []
    print("First we iterate thorough the sequence dict of the json and for each 'Extude_Feature' we create an 'Extrude' object.\n")
    for item in json_data["sequence"]:
        print(item)
        if item["type"] == "ExtrudeFeature":
            extrude_ops = get_Extrude(all_stat, item["entity"])

In [178]:
def get_Extrude(all_stat, extrude_id, sketch_dim=256):
    
    extrude_entity = all_stat["entities"][extrude_id]
    assert extrude_entity["start_extent"]["type"] == "ProfilePlaneStartDefinition"
    all_skets = []
    n = len(extrude_entity["profiles"])

    print(f"\nTo create the 'Extrude' object for {extrude_id} we calculate the number of profiles it contains: {n}\n")

    for i in range(len(extrude_entity["profiles"])):
        sket_id, profile_id = extrude_entity["profiles"][i]["sketch"], extrude_entity["profiles"][i]["profile"]
        print(f"We then extract the sketch id of the profile: {sket_id}, and the profile id: {profile_id}\n")
        sket_entity = all_stat["entities"][sket_id]
        print(f"Then we create a 'Profile' class by providing the profile id {profile_id} to the Profile constructor.\n")
        
        sket_profile = get_Profile(sket_entity["profiles"][profile_id])

In [179]:
def get_Profile(stat):
    print(f"A 'Profile' class is a list of 'Loop' classes, therefore to construct a 'Profile' we have to sequentially "
          f"create 'Loop' objects defined in the json data. Each sketch is made up of a 'Profile' for which we provided the id. "
          f"Under the profile id we can see the loops. Each 'Loop' has a 'is_outer' bool and a number of profile curves. These "
          f"will be provided to the 'Curve' constructor.\n")
    print(f"Number of loops in this Profile: {len(stat['loops'])}")
    all_loops = [get_Loop(item) for item in stat['loops']]
    return Profile(all_loops)

In [180]:
def get_Loop(stat):
    print(f"\nThe 'Loop' class consists of a list of 'Curve' classes. The 'Loop' constructor iterates through the "
          f"curve information provided in the json data.")
    for i, item in enumerate(stat['profile_curves']):
        print(i, item)

    all_curves = [get_Curve(item) for item in stat['profile_curves']]
    this_loop = Loop(all_curves)
    this_loop.is_outer = stat['is_outer']

In [181]:
def get_Curve(stat):
    print(f"\nDepending on the type of 'Curve' (Arc, Line, Circle) call the respective constructor.\n")
    print(f"Type: {stat['type']}")
    if stat['type'] == "Line3D":
        return get_Line(stat)
    elif stat['type'] == "Circle3D":
        return get_Circle(stat)
    elif stat['type'] == "Arc3D":
        return get_Arc(stat)
    else:
        raise NotImplementedError("curve type not supported yet: {}".format(stat['type']))

In [186]:
def get_Line(stat):
    assert stat['type'] == "Line3D"
    print(f"To create a 'Line' object, we extract the start and endpoint of the line as numpy arrays.")
    start_point = np.array([stat['start_point']['x'],
                            stat['start_point']['y']])
    end_point = np.array([stat['end_point']['x'],
                          stat['end_point']['y']])
    print(f"Start point: {start_point}\nEnd point: {end_point}")
    return Line(start_point, end_point)

def get_Circle(stat):
    print(f"To create a 'Circle' object, we extract the center point, the radius and the normal.")
    assert stat['type'] == "Circle3D"
    center = np.array([stat['center_point']['x'],
                       stat['center_point']['y']])
    radius = stat['radius']
    normal = np.array([stat['normal']['x'],
                       stat['normal']['y'],
                       stat['normal']['z']])
    print(f"Center: {center}, Radius: {radius}, Normal: {normal}")
    return Circle(center, radius, normal)

def get_Arc(stat):
    print(f"To create a 'Arc' object, we extract the start-, center- and end-point, the radius, the normal, "
          f"the start- and eng-angle and the reference vector.")
    assert stat['type'] == "Arc3D"
    start_point = np.array([stat['start_point']['x'],
                            stat['start_point']['y']])
    end_point = np.array([stat['end_point']['x'],
                          stat['end_point']['y']])
    center = np.array([stat['center_point']['x'],
                       stat['center_point']['y']])
    radius = stat['radius']
    normal = np.array([stat['normal']['x'],
                       stat['normal']['y'],
                       stat['normal']['z']])
    start_angle = stat['start_angle']
    end_angle = stat['end_angle']
    ref_vec = np.array([stat['reference_vector']['x'],
                        stat['reference_vector']['y']])
    print(f"Start point: {start_point}\nEnd point: {end_point}\nCenter: {center}\nRadius: {radius}\nNormal: {normal}\n"
          f"Start angle: {start_angle}\nEnd angle: {end_angle}\nReference vector: {ref_vec}\n")
    return Arc(start_point, end_point, center, radius, normal, start_angle, end_angle, ref_vec)

In [187]:
get_CADSequence(json_data)

First we iterate thorough the sequence dict of the json and for each 'Extude_Feature' we create an 'Extrude' object.

{'index': 0, 'type': 'Sketch', 'entity': 'FZju8KWrsx7pSQP_0'}
{'index': 1, 'type': 'ExtrudeFeature', 'entity': 'FK8QzQJB7rehzNa_0'}

To create the 'Extrude' object for FK8QzQJB7rehzNa_0 we calculate the number of profiles it contains: 1

We then extract the sketch id of the profile: FZju8KWrsx7pSQP_0, and the profile id: JGK

Then we create a 'Profile' class by providing the profile id JGK to the Profile constructor.

A 'Profile' class is a list of 'Loop' classes, therefore to construct a 'Profile' we have to sequentially create 'Loop' objects defined in the json data. Each sketch is made up of a 'Profile' for which we provided the id. Under the profile id we can see the loops. Each 'Loop' has a 'is_outer' bool and a number of profile curves. These will be provided to the 'Curve' constructor.

Number of loops in this Profile: 3

The 'Loop' class consists of a list of 'C

AttributeError: 'NoneType' object has no attribute 'bbox'